7. Extend the gold layer with a second aggregation for a different stakeholder (e.g., the inventory team)
and justify why it belongs in gold rather than being computed ad hoc by that team.

## Second Gold Aggregation – Inventory Team

I extended the Gold layer by creating an inventory-focused aggregation for the
inventory team.

We are calculating total quantity sold and total number of orders for each product. This helps the inventory team understand which products are selling more and may need more stock.

```
SELECT
product_id,
SUM(quantity) AS total_quantity,
SUM(total_amount) AS total_revenue
FROM
cyntexa_dev.silver.sales_silver
GROUP BY
product_id
ORDER BY
total_revenue DESC;
```

I kept this aggregation in the Gold layer because it is a reusable,
business-ready metric. Instead of the inventory team calculating the same
metrics manually from Silver data every time, they can directly use this Gold
view. This also keeps the business logic consistent and avoids repeated
processing.

**why kept all this into gold**

Instead of every team or individual, such as the inventory team, business
owners, or other stakeholders, processing the Silver tables themselves to
calculate the same aggregation, we keep this business-ready aggregation in
the Gold layer.

This avoids repeated and time-consuming processing on the Silver data and
reduces unnecessary compute usage. It also prevents different teams from
using slightly different logic and getting inconsistent results.

By keeping the aggregation in Gold, everyone can use the same trusted and
centralized data source. This makes reporting simpler, faster, and more
consistent across the organization.


8.Write a short design note on which parts of this pipeline should run in the customer's data plane vs.
rely on Databricks' control plane, and what that means for a network/security review.

**Design Note – Customer Data Plane vs Databricks Control Plane**

For this sales pipeline, the actual data processing such as reading the CSV,
cleaning the Silver data, and creating Gold aggregations should run in the
**customer's data plane**. This keeps the customer's data and processing within
the customer's cloud environment.

The **Databricks control plane** should mainly be used for managing the
workspace, scheduling and orchestrating jobs, and controlling configurations
and permissions.

**Network and Security Considerations**

For the security review, the team should verify how the data plane connects to
the customer's storage and other data sources. Network access, firewall rules,
private connectivity, IAM permissions, encryption, and audit logging should be
checked to make sure data is properly protected.

In simple terms:

**Control Plane → manages and orchestrates the pipeline**

**Customer Data Plane → runs the actual data processing**

This separation helps keep customer data protected while still using
Databricks to manage and automate the pipeline.

9. (Data Analyst) Build a query or lightweight dashboard directly against the gold table, and identify one data-quality issue you can trace back to a specific bronze or silver transformation decision.

# Gold Dashboard and Data Quality Issue

I created a lightweight dashboard using the Gold table
`cyntexa_dev.gold.product_gold` to show product-level metrics such as
**total quantity sold** and **total revenue**.

I also used the Silver table `cyntexa_dev.silver.sales_silver` to compare the
detailed cleaned data with the Gold aggregation.

During the analysis, I identified a potential data-quality issue in the Silver
transformation:

`dropDuplicates(["order_id"])`

This can remove records when multiple valid rows have the same `order_id`.
As a result, the Gold aggregation may show lower total quantity or revenue
than expected.

### Data Flow

**Bronze → Silver (`dropDuplicates`) → Gold (`product_gold`) → Dashboard**

This shows how a data-quality issue observed in the Gold dashboard can be
traced back to a specific transformation decision in the Silver layer.